In [3]:
import os

base = "../MAGFiLO_1.0_Kaggle_2026" 

print(os.listdir(base))
print(os.listdir(os.path.join(base, "train")))
print(os.listdir(os.path.join(base, "train", "train_images"))[:5])  # just first 5, folder is big

['train', 'test']
['train_images', 'MAGFiLO_1.0_Annotations_kaggle2026_train.json']
['20130219063134Lh.jpeg', '20140813064254Uh.jpeg', '20150203084714Th.jpeg', '20160317232434Lh.jpeg', '20120227161654Bh.jpeg']


In [5]:
from PIL import Image
import numpy as np

img_path = os.path.join(base, "train", "train_images", "20130219063134Lh.jpeg")
img = Image.open(img_path)

print("Mode:", img.mode)        # expect 'L' (grayscale) or possibly 'RGB' if saved oddly
print("Size:", img.size)        # expect (2048, 2048)

img_array = np.array(img)
print("Array shape:", img_array.shape)  
print("Value range:", img_array.min(), img_array.max())

Mode: L
Size: (2048, 2048)
Array shape: (2048, 2048)
Value range: 0 255


In [7]:
import json

ann_path = os.path.join(base, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")  # adjust filename if different

with open(ann_path, "r") as f:
    annotations_data = json.load(f)

# First, just see the shape of the data — don't assume COCO format yet
print("Top-level keys:", list(annotations_data.keys()) if isinstance(annotations_data, dict) else "list, not dict")
if isinstance(annotations_data, dict):
    for k, v in annotations_data.items():
        print(f"  {k}: {type(v)}, len={len(v) if hasattr(v, '__len__') else 'n/a'}")

Top-level keys: ['info', 'licenses', 'categories', 'images', 'annotations']
  info: <class 'dict'>, len=9
  licenses: <class 'dict'>, len=3
  categories: <class 'list'>, len=4
  images: <class 'list'>, len=1154
  annotations: <class 'list'>, len=8199


In [9]:
print("Categories:")
for c in annotations_data["categories"]:
    print(" ", c)

print("\nSample image entry:")
print(annotations_data["images"][0])

print("\nSample annotation entry:")
print(annotations_data["annotations"][0])

# Also check: are there really repeated (image, different annotator) entries?
# quick check relevant to your Sep 7-8 task later
from collections import Counter
img_id_counts = Counter(a["image_id"] for a in annotations_data["annotations"])
print("\nMax annotations for a single image_id:", max(img_id_counts.values()))
print("Number of image_ids with >1 annotation:", sum(1 for v in img_id_counts.values() if v > 1))

Categories:
  {'supercategory': 'filament', 'id': 1, 'name': 'Left'}
  {'supercategory': 'filament', 'id': 2, 'name': 'Right'}
  {'supercategory': 'filament', 'id': 3, 'name': 'Unidentifiable'}
  {'supercategory': 'filament', 'id': 4, 'name': 'Ambiguous'}

Sample image entry:
{'license': 1, 'file_name': '20140609195854Bh.jpeg', 'url': 'https://gong2.nso.edu/HA/hag/201406/20140609/20140609195854Bh.jpg', 'height': 2048, 'width': 2048, 'date_captured': '2014-06-09 19:58:54', 'id': '040301-20140609195854Bh'}

Sample annotation entry:
{'segmentation': [[480.9382, 850.955, 480.307, 852.0535, 478.2989, 852.451, 476.1202, 851.3087, 471.8813, 845.6188, 469.2107, 844.1978, 460.0, 845.0, 459.0, 844.0, 455.9884, 843.2471, 452.2964, 839.9584, 450.6544, 839.7575, 449.7991, 838.6671, 450.223, 837.6258, 447.5581, 834.6278, 445.4449, 829.1312, 442.9577, 825.6077, 441.8277, 824.4777, 438.6437, 823.5973, 439.0, 821.0, 436.0, 818.0, 435.1557, 815.4672, 433.9753, 817.121, 431.9582, 816.9118, 431.3678, 815.

In [10]:
import json
import os

def load_annotations(json_path):
    """
    Loads MAGFiLO COCO-style annotations.

    Returns:
        images: dict mapping image_id (str) -> image info dict
                {file_name, width, height, date_captured, ...}
        annotations_by_image: dict mapping image_id (str) -> list of annotation dicts
                (NOT deduplicated — every annotation entry is kept, including
                 repeated ones for the same image, per the Sep 7-8 task)
        categories: dict mapping category_id (int) -> category name
    """
    with open(json_path, "r") as f:
        data = json.load(f)

    images = {img["id"]: img for img in data["images"]}

    categories = {c["id"]: c["name"] for c in data["categories"]}

    annotations_by_image = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        annotations_by_image.setdefault(img_id, []).append(ann)

    return images, annotations_by_image, categories


# --- Load it ---
json_path = os.path.join(base, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
images, annotations_by_image, categories = load_annotations(json_path)

print("Num images:", len(images))
print("Num image_ids with annotations:", len(annotations_by_image))
print("Categories:", categories)

# --- Sanity check: every annotated image_id actually exists in images dict ---
missing = [iid for iid in annotations_by_image if iid not in images]
print("Annotation image_ids missing from images dict:", len(missing))

# --- Sanity check: every image file referenced actually exists on disk ---
img_dir = os.path.join(base, "train", "train_images")
missing_files = [
    info["file_name"] for info in images.values()
    if not os.path.exists(os.path.join(img_dir, info["file_name"]))
]
print("Referenced image files missing on disk:", len(missing_files))

# --- Convenience: pull image + its annotations together ---
def get_sample(image_id):
    img_info = images[image_id]
    img_path = os.path.join(img_dir, img_info["file_name"])
    anns = annotations_by_image.get(image_id, [])
    return img_info, img_path, anns

# quick test
some_id = list(images.keys())[0]
info, path, anns = get_sample(some_id)
print(f"\nSample: {info['file_name']}, {len(anns)} filament instance(s)")

Num images: 1154
Num image_ids with annotations: 1154
Categories: {1: 'Left', 2: 'Right', 3: 'Unidentifiable', 4: 'Ambiguous'}
Annotation image_ids missing from images dict: 0
Referenced image files missing on disk: 0

Sample: 20140609195854Bh.jpeg, 12 filament instance(s)


In [14]:
all_keys = set()
for ann in annotations_data["annotations"]:
    all_keys.update(ann.keys())
print(all_keys)

print("Total annotation entries (instance-level samples):", len(annotations_data["annotations"]))  # 8199
print("Total unique images:", len(annotations_data["images"]))  # 1154
print("Average instances per image:", 8199 / 1154)

from collections import Counter
seg_strs = [str(ann["segmentation"]) for ann in annotations_data["annotations"]]
dupe_counts = Counter(seg_strs)
exact_dupes = sum(1 for c in dupe_counts.values() if c > 1)
print("Exact duplicate segmentations:", exact_dupes)

{'iscrowd', 'category_id', 'bbox', 'id', 'spine', 'segmentation', 'area', 'image_id'}
Total annotation entries (instance-level samples): 8199
Total unique images: 1154
Average instances per image: 7.104852686308492
Exact duplicate segmentations: 0
